# IndicF5 safety test (does NOT change the ReVoice server)

Checks that **IndicF5** (AI4Bharat's F5-TTS fine-tune for 11 Indian languages) can run in the
same Colab session as everything ReVoice already uses, **without breaking it**:
1. Runs the server notebook's install and load cells **unchanged** (Steps 1 – 4a).
2. Makes one **Hindi** sentence from an **English** reference clip (cross-lingual) with IndicF5.
3. Checks XTTS, F5-TTS, Whisper and SpeechBrain still work, and that no package version changed.
4. **Step T7** tries settings that remove the garbled start of the Hindi sentence (method B only).

**Why we do not `pip install` IndicF5's GitHub code:** that package is also called `f5_tts`
(version 0.1.0), so it would replace our f5-tts 1.1.22, and it asks for `transformers<4.50` and
`numpy<=1.26.4`, which would break the pinned versions from Step 2. Instead we load IndicF5's
weights with the f5-tts that is already installed, plus a tiny compatibility fix (Step T3).

**Before starting:** Runtime → Change runtime type → **T4 GPU**; accept the IndicF5 terms on
Hugging Face; add the `HF_TOKEN` secret (see Step T1).

Run cells **top to bottom, once each**. Copy the block printed by **Step T6**, and later the one
printed by **Step T7**, back to Claude.

**Tested (Nov 2026) on Colab's Python 3.13 + torch 2.11:** Steps 2b–T6 all passed, nothing changed
versions; method B works, method A fails with a meta-tensor error.

Licence note: F5-TTS and IndicF5 weights are for **non-commercial use** (fine for a college project).

## Step 1 — Check the GPU  *(copied unchanged from revoice_server_v2)*

In [ ]:
!nvidia-smi

## Step 2 — Install everything (3–5 minutes)
Installs XTTS, then F5-TTS **without letting it change other packages**, then ONE final line pins the known-working versions, then a conflict check (diffusers / gradio warnings can be ignored: ReVoice doesn't use them).
**After this finishes: Runtime → Restart session**, then continue from Step 2b.

In [ ]:
# XTTS + server tools
!pip install -q coqui-tts fastapi uvicorn pyngrok soundfile

# F5-TTS: --no-deps so it can't overwrite transformers/numpy
!pip install -q --no-deps f5-tts cached_path
!pip install -q ema_pytorch torchdiffeq vocos x_transformers hydra-core omegaconf jieba pypinyin unidecode pydub wandb
!pip install -q rjieba || echo "rjieba optional - skipped"

# SpeechBrain (metrics only): --no-deps so it can't change transformers/numpy/tokenizers/huggingface-hub.
!pip install -q --no-deps speechbrain hyperpyyaml "ruamel.yaml>=0.17.28,<0.19" ruamel.yaml.clib

# ffmpeg converts the browser's webm/opus recordings
!apt-get -qq install -y ffmpeg > /dev/null

# ONE final pinning line with the known-working versions; it must stay LAST so nothing changes them after.
!pip install -q tomli transformers_stream_generator boto3 bitsandbytes "transformers==4.57.6" "huggingface-hub>=0.34.0,<1.0" "tokenizers>=0.22.0,<=0.23.0" "numpy<2.3"

# Conflict check for the packages ReVoice uses (diffusers/gradio warnings are expected and ignored).
!pip check 2>&1 | grep -iE "transformers|coqui|tts|f5|librosa|numba|torch|cached|speechbrain|hyperpyyaml" | grep -viE "^(diffusers|gradio)" || echo "No conflicts in the packages ReVoice uses."

print("Install finished. Now: Runtime -> Restart session, then run Step 2b.")

## Step 2b — Check versions (after restarting)
If this errors, don't continue.

In [ ]:
import transformers, numpy, librosa, tokenizers, huggingface_hub
from packaging.version import Version
from transformers.pytorch_utils import isin_mps_friendly
from transformers.utils.import_utils import is_torch_greater_or_equal
print("transformers   ", transformers.__version__)
print("tokenizers     ", tokenizers.__version__)
print("huggingface-hub", huggingface_hub.__version__)
print("numpy          ", numpy.__version__)

# compare with the known-working set from Step 2
problems = []
if transformers.__version__ != "4.57.6":
    problems.append(f"transformers is {transformers.__version__}, expected 4.57.6")
if not Version("0.22.0") <= Version(tokenizers.__version__) <= Version("0.23.0"):
    problems.append(f"tokenizers is {tokenizers.__version__}, expected 0.22.0 to 0.23.0")
if not Version("0.34.0") <= Version(huggingface_hub.__version__) < Version("1.0"):
    problems.append(f"huggingface-hub is {huggingface_hub.__version__}, expected 0.34 up to (not incl.) 1.0")
if Version(numpy.__version__) >= Version("2.3"):
    problems.append(f"numpy is {numpy.__version__}, expected below 2.3")

import f5_tts.api
print("f5-tts importable")
import speechbrain
print("speechbrain    ", speechbrain.__version__, "(for the voice metrics)")
if problems:
    raise RuntimeError("Wrong versions:\n - " + "\n - ".join(problems) +
                       "\nRe-run Step 2, then Runtime -> Restart session.")
print("All checks passed")

## Step 3 — Load XTTS-v2 (~2 GB download first time)

In [ ]:
import os
os.environ["COQUI_TOS_AGREED"] = "1"

import torch
from TTS.api import TTS

device = "cuda" if torch.cuda.is_available() else "cpu"
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
xtts_model = tts.synthesizer.tts_model
SAMPLE_RATE = 24000
print("XTTS-v2 loaded on", device)

## Step 3b — Load F5-TTS (~1.5 GB download first time)
Also warms up Whisper (the speech-to-text model F5 uses for the reference transcript), so it downloads now and not during your first request.
If this fails, XTTS still works — the app will report the F5 error when you pick it.

In [ ]:
f5 = None
try:
    from f5_tts.api import F5TTS
    f5 = F5TTS(device=device)
    print("F5-TTS loaded")
except Exception as e:
    import traceback; traceback.print_exc()
    print("\nF5-TTS failed to load (XTTS still available):", e)

if f5 is not None:
    # Warm-up downloads Whisper (~1.5 GB) now, so the first real request doesn't time out.
    try:
        from importlib.resources import files
        example = str(files("f5_tts").joinpath("infer/examples/basic/basic_ref_en.wav"))
        print("Whisper warm-up transcript:", f5.transcribe(example))
    except Exception as e:
        print("Whisper warm-up failed (F5 will try again on the first request):", e)

## Step 3c — Load the evaluation models (small, ~100 MB)
Used only to MEASURE results: speaker similarity (SpeechBrain ECAPA-TDNN) and an English accent estimate (SpeechBrain CommonAccent). If they fail to load, speech still works and those numbers show as n/a.

In [ ]:
# SpeechBrain models used only to measure results: speaker similarity and accent (CommonAccent).
# If one fails to load, speech still works and that metric shows n/a.
spk_model = None
accent_model = None

try:
    from speechbrain.inference.speaker import SpeakerRecognition
    spk_model = SpeakerRecognition.from_hparams(
        source="speechbrain/spkrec-ecapa-voxceleb",
        savedir="/content/metric_models/spkrec-ecapa-voxceleb",
        run_opts={"device": device})
    print("Speaker similarity model loaded")
except Exception as e:
    import traceback; traceback.print_exc()
    print("Speaker similarity model failed to load (similarity will be n/a):", e)

try:
    from speechbrain.inference.classifiers import EncoderClassifier
    accent_model = EncoderClassifier.from_hparams(
        source="Jzuluaga/accent-id-commonaccent_ecapa",
        savedir="/content/metric_models/accent-id-commonaccent_ecapa",
        run_opts={"device": device})
    print("Accent model loaded")
except Exception as e:
    import traceback; traceback.print_exc()
    print("Accent model failed to load (accent will be n/a):", e)

## Step 4a — Load the ReVoice voice functions + metrics
The code that turns text + reference recordings into speech, and measures each result (similarity, pitch, speaking rate, accent). The API in Step 6 and the test in Step 4b both use it.

In [ ]:
# Voice functions shared by the Step 4b test and the Step 6 API, so the test runs the same code.
import os, base64, hashlib, subprocess, tempfile, time
import soundfile as sf
from types import SimpleNamespace
try:
    import f5_tts.infer.utils_infer as f5_utils     # F5's own reference trimming + caches
except Exception:
    f5_utils = None                                 # F5 not installed: XTTS still works

xtts_cache = {}   # voice profiles for XTTS
f5_cache = {}     # (trimmed reference clip, its transcript) for F5
temp_files = []   # cleaned clips on disk, deleted by forget_all_voices()

# XTTS: emotion changes speaking speed and how "lively" it sounds
EMOTION_STYLE = {
    "neutral": {"speed": 1.00, "temperature": 0.80},
    "happy":   {"speed": 1.05, "temperature": 0.85},
    "urgent":  {"speed": 1.12, "temperature": 0.85},
    "tender":  {"speed": 0.92, "temperature": 0.75},
}

# F5 settings
F5_SPEED = 1.0              # always normal speed: the reference recording carries the emotion
F5_NFE_STEPS = 64           # more refinement steps = clearer speech (32 was a bit rough)
F5_MAX_REF_SECONDS = 15     # pass up to 15s; F5 then trims to <=12s at a pause by itself

def clean_clip(b64, max_seconds=None):
    """Decode a browser recording and clean it for cloning."""
    raw = base64.b64decode(b64)
    f = tempfile.NamedTemporaryFile(suffix=".input", delete=False)
    f.write(raw); f.close()
    out = f.name + ".wav"
    cmd = ["ffmpeg", "-y", "-i", f.name, "-ac", "1", "-ar", "24000",
           "-af", "highpass=f=70,"
                  "silenceremove=start_periods=1:start_silence=0.1:start_threshold=-45dB,"
                  "loudnorm=I=-18:TP=-2:LRA=11"]
    if max_seconds:
        cmd += ["-t", str(max_seconds)]
    r = subprocess.run(cmd + [out], capture_output=True, text=True)
    os.remove(f.name)              # the raw upload isn't needed any more
    if r.returncode != 0:
        raise RuntimeError("ffmpeg failed:\n" + r.stderr[-1200:])
    temp_files.append(out)
    return out

def usable_clips(clips_b64, max_seconds=None):
    out = []
    for b64 in clips_b64:
        p = clean_clip(b64, max_seconds)
        d = sf.info(p).duration
        if d >= 0.8:
            out.append((p, d))
    if not out:
        raise RuntimeError("All reference clips were too short or silent. Re-record and keep talking.")
    return out

def clips_key(clips_b64):
    return hashlib.md5("".join(clips_b64).encode()).hexdigest()

# XTTS cloned from the user's recordings
def run_xtts(clips_b64, req, style):
    key = clips_key(clips_b64)
    if key not in xtts_cache:
        clips = usable_clips(clips_b64)
        print(f"   XTTS profile from {sum(d for _, d in clips):.1f}s of audio")
        xtts_cache[key] = xtts_model.get_conditioning_latents(
            audio_path=[p for p, _ in clips], gpt_cond_len=30, gpt_cond_chunk_len=4, max_ref_length=30)
    gpt_cond_latent, speaker_embedding = xtts_cache[key]
    out = xtts_model.inference(
        req.text, req.language, gpt_cond_latent, speaker_embedding,
        temperature=style["temperature"], speed=style["speed"],
        length_penalty=1.0, repetition_penalty=10.0, top_k=50, top_p=0.85,
        enable_text_splitting=True)
    return out["wav"], SAMPLE_RATE, {}

# XTTS stock (built-in) voice
def stock_speaker_names():
    return sorted(xtts_model.speaker_manager.speaker_names)

def run_xtts_stock(req, style):
    speakers = xtts_model.speaker_manager.speakers
    if req.stock_speaker not in speakers:
        raise RuntimeError(f"Unknown stock voice {req.stock_speaker!r}. "
                           f"Some valid ones: {', '.join(stock_speaker_names()[:8])}")
    # built-in voices come with their voice profile already computed
    gpt_cond_latent, speaker_embedding = speakers[req.stock_speaker].values()
    out = xtts_model.inference(
        req.text, req.language, gpt_cond_latent, speaker_embedding,
        temperature=style["temperature"], speed=style["speed"],
        length_penalty=1.0, repetition_penalty=10.0, top_k=50, top_p=0.85,
        enable_text_splitting=True)
    return out["wav"], SAMPLE_RATE, {}

# F5-TTS
def run_f5(clips_b64, req):
    if f5 is None:
        raise RuntimeError("F5-TTS isn't loaded. Check the output of Step 3b.")
    key = clips_key(clips_b64)
    if key not in f5_cache:
        # F5 uses ONE reference: the longest clip, up to 15 seconds
        clips = usable_clips(clips_b64, max_seconds=F5_MAX_REF_SECONDS)
        longest = max(clips, key=lambda c: c[1])[0]
        # F5 trims first (<=12 s, at a pause); the empty transcript makes Whisper transcribe the trimmed
        # audio, so the words match what F5 hears (a mismatch causes mumbled or skipped words).
        ref_path, ref_text = f5_utils.preprocess_ref_audio_text(longest, "", show_info=lambda *a, **k: None)
        print(f"   F5 reference: {sf.info(ref_path).duration:.1f}s, transcript: {ref_text!r}")
        f5_cache[key] = (ref_path, ref_text)
    ref_path, ref_text = f5_cache[key]
    wav, sr, _ = f5.infer(ref_file=ref_path, ref_text=ref_text, gen_text=req.text,
                          speed=F5_SPEED, nfe_step=F5_NFE_STEPS, show_info=lambda *a, **k: None)
    return wav, sr, {"ref_text": ref_text.strip()}

# Voice metrics for the reference and the output: speaker similarity (ECAPA cosine, 1 = same voice),
# pitch, speaking rate, and accent as top-3 raw scores (not probabilities). A failed metric is None.
import numpy as np, librosa, torch

METRICS_ENABLED = True       # set False to skip metrics (makes each request ~1-3 s faster)
METRIC_SR = 16000            # both SpeechBrain models expect 16 kHz audio
reference_metrics_cache = {}

# keep models loaded in Step 3c (None if that step was skipped or failed)
spk_model = globals().get("spk_model")
accent_model = globals().get("accent_model")

def safe(name, fn, *args):
    """Runs one metric; on any error prints it and returns None instead of crashing."""
    try:
        return fn(*args)
    except Exception as e:
        print(f"   metric '{name}' failed: {type(e).__name__}: {e}")
        return None

def speaker_embedding(wav16):
    with torch.no_grad():
        return spk_model.encode_batch(torch.tensor(wav16).float()).squeeze().cpu()

def pitch_stats(wav16):
    f0, voiced, _ = librosa.pyin(wav16, fmin=65, fmax=400, sr=METRIC_SR, frame_length=1024)
    f0 = f0[voiced & ~np.isnan(f0)]
    if len(f0) < 10:
        return None, None
    return round(float(np.mean(f0)), 1), round(float(np.percentile(f0, 95) - np.percentile(f0, 5)), 1)

def speech_seconds_of(wav16):
    """Seconds of actual speech (pauses removed)."""
    parts = librosa.effects.split(wav16, top_db=35)
    return sum(e - s for s, e in parts) / METRIC_SR

def words_per_second(wav16, n_words):
    if not n_words:
        return None
    speech_seconds = speech_seconds_of(wav16)
    return round(n_words / speech_seconds, 2) if speech_seconds > 0.3 else None

def accent_scores(wav16):
    """The 3 most likely accents with their RAW scores: [[label, score], ...], best first.
    This model's outputs are cosine scores (-1..1), NOT probabilities, so we pass them on
    as they are. (Before v8 they went through x30 + softmax: an unconfirmed scale that
    showed e.g. "US 100%" when US was only 0.2 ahead.)"""
    with torch.no_grad():
        out, _, _, _ = accent_model.classify_batch(torch.tensor(wav16).float().unsqueeze(0))
    scores = out.squeeze(0).float().cpu()
    labels = accent_model.hparams.label_encoder.ind2lab
    best = torch.argsort(scores, descending=True)[:3]
    return [[labels[int(i)], round(float(scores[i]), 3)] for i in best]

def voice_stats(wav16, n_words):
    """Pitch, rate and accent for one piece of audio (each None if it can't be measured)."""
    pitch = safe("pitch", pitch_stats, wav16) or (None, None)
    return {"pitch_hz": pitch[0], "pitch_range_hz": pitch[1],
            "words_per_sec": safe("rate", words_per_second, wav16, n_words),
            "speech_seconds": safe("speech", lambda: round(speech_seconds_of(wav16), 2)),
            "accent_scores": safe("accent", accent_scores, wav16) if accent_model is not None else None}

def reference_metrics(clips_b64):
    """Measures the real voice once per set of clips (cached): up to 30 s of the cleaned clips."""
    key = clips_key(clips_b64)
    if key not in reference_metrics_cache:
        clips = usable_clips(clips_b64)
        wav16 = np.concatenate([librosa.load(p, sr=METRIC_SR, mono=True)[0] for p, _ in clips])[:30 * METRIC_SR]
        # words in the reference: Whisper writes down what was said (needs F5's Whisper)
        n_words = None
        if f5 is not None:
            text = safe("transcribe", lambda: " ".join(f5.transcribe(p) for p, _ in clips))
            n_words = len(text.split()) if text else None
        reference_metrics_cache[key] = {
            "stats": voice_stats(wav16, n_words),
            "embedding": safe("embedding", speaker_embedding, wav16) if spk_model is not None else None,
        }
    return reference_metrics_cache[key]

def compute_metrics(req, clips_b64, wav, sr):
    """All metrics for one generated sentence. Stock voices have no reference recording,
    so only the output is measured for them."""
    out16 = librosa.resample(np.asarray(wav, dtype=np.float32), orig_sr=sr, target_sr=METRIC_SR)
    metrics = {"similarity": None, "reference": None, "accent_match": None,
               "output": voice_stats(out16, len(req.text.split()))}
    if clips_b64:
        ref = safe("reference", reference_metrics, clips_b64)
        if ref:
            metrics["reference"] = ref["stats"]
            if ref["embedding"] is not None:
                out_emb = safe("embedding", speaker_embedding, out16)
                if out_emb is not None:
                    metrics["similarity"] = round(float(
                        torch.nn.functional.cosine_similarity(ref["embedding"], out_emb, dim=0)), 3)
            # accent_match is left to the backend/page (needs enough speech and a clear winner on both sides)
    return metrics

# Single entry point, used by the API and by Step 4b
def generate(req, clips):
    """Makes the speech. Returns (wav, sample_rate, extra_info)."""
    style = EMOTION_STYLE.get(req.emotion, EMOTION_STYLE["neutral"])
    if req.stock_speaker:
        # stock voice: no recordings needed (XTTS only)
        if req.engine != "xtts":
            raise RuntimeError("Stock voices only work with XTTS-v2.")
        print(f"\n--- [xtts stock] {req.text!r} | {req.emotion} | {req.stock_speaker}")
        return run_xtts_stock(req, style)
    if not clips:
        raise RuntimeError("No reference audio sent.")
    print(f"\n--- [{req.engine}] {req.text!r} | {req.emotion} | {len(clips)} clip(s)")
    if req.engine == "f5":
        return run_f5(clips, req)
    if req.engine == "xtts":
        return run_xtts(clips, req, style)
    raise RuntimeError(f"Unknown engine {req.engine!r}.")

def synthesize_core(req):
    """Returns (wav, sample_rate, extra_info). req needs: text, language, emotion,
    engine, stock_speaker, speaker_wav_base64, speaker_wav_base64_list.
    extra_info has synth_seconds, metrics, metrics_seconds (and ref_text for F5)."""
    clips = [] if req.stock_speaker else (
        req.speaker_wav_base64_list or ([req.speaker_wav_base64] if req.speaker_wav_base64 else []))
    t0 = time.time()
    wav, sr, extra = generate(req, clips)
    extra["synth_seconds"] = round(time.time() - t0, 2)      # speech generation only

    extra["metrics"] = None
    if METRICS_ENABLED:
        t1 = time.time()
        extra["metrics"] = safe("metrics", compute_metrics, req, clips, wav, sr)
        extra["metrics_seconds"] = round(time.time() - t1, 2)
        print(f"   metrics: {extra['metrics']}")
    return wav, sr, extra

def forget_all_voices():
    """Wipes every cached voice and cleaned clip (called when a user is deleted)."""
    n = len(xtts_cache) + len(f5_cache)
    xtts_cache.clear(); f5_cache.clear()
    # F5 keeps its own cache of trimmed clips and transcripts: wipe that too
    if f5_utils is not None:
        temp_files.extend(f5_utils._ref_audio_cache.values())
        f5_utils._ref_audio_cache.clear(); f5_utils._ref_text_cache.clear()
    for path in temp_files:
        try:
            os.remove(path)
        except OSError:
            pass
    temp_files.clear()
    print(f"\n--- forgot {n} cached voice(s)")
    return n

print("Voice functions ready.")

## Step T1 — Hugging Face token (IndicF5 is gated)
One-time setup:
1. Open https://huggingface.co/ai4bharat/IndicF5 while logged in, and accept the terms.
2. https://huggingface.co/settings/tokens → **Create new token** → type **Read** → copy it.
3. In Colab: **key icon** (left bar) → **Add new secret** → name `HF_TOKEN`, value = the token,
   switch **Notebook access** on.

In [ ]:
# Reads the Hugging Face token from Colab secrets and checks we may download IndicF5.
import os, sys, json, time, traceback, platform
from google.colab import userdata
from huggingface_hub import login, HfApi, hf_hub_download
from huggingface_hub.utils import GatedRepoError, RepositoryNotFoundError
from importlib.metadata import version as pkg_version, PackageNotFoundError

INDICF5_REPO = "ai4bharat/IndicF5"
RESULTS = {}      # everything Step T6 prints at the end

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:
    raise RuntimeError("No HF_TOKEN secret. Key icon (left bar) -> Add new secret -> name HF_TOKEN, "
                       "value = your Hugging Face token, and switch 'Notebook access' on.") from e
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
print("Logged in to Hugging Face as:", HfApi().whoami()["name"])

try:
    hf_hub_download(INDICF5_REPO, "config.json")
    INDICF5_FILES = HfApi().list_repo_files(INDICF5_REPO)
except GatedRepoError as e:
    raise RuntimeError("Access to IndicF5 not granted yet. Open https://huggingface.co/ai4bharat/IndicF5, "
                       "accept the terms, then re-run this cell. (Fine-grained tokens also need "
                       "'Read access to public gated repos'.)") from e
print("IndicF5 access OK. Files:", INDICF5_FILES)

# package versions BEFORE IndicF5, to prove nothing changes
WATCH = ["torch", "torchaudio", "transformers", "tokenizers", "huggingface-hub", "numpy", "librosa",
         "numba", "f5-tts", "coqui-tts", "speechbrain", "safetensors", "accelerate", "vocos",
         "x-transformers", "pydub", "rjieba", "jieba", "pypinyin"]
def versions():
    out = {}
    for p in WATCH:
        try:
            out[p] = pkg_version(p)
        except PackageNotFoundError:
            out[p] = "-"
    return out
VERSIONS_BEFORE = versions()
RESULTS["python"] = platform.python_version()
RESULTS["versions"] = VERSIONS_BEFORE
for k, v in VERSIONS_BEFORE.items():
    print(f"   {k:16} {v}")

## Step T2 — English reference clip + baseline of the current engines
Upload an **English** recording of your voice (6–20 s). Set `USE_MY_CLIP = False` to use F5's
built-in example voice instead. XTTS and F5 then speak the Step 4b sentence through the exact
API code, so we can compare with the same test **after** IndicF5 is loaded.

In [ ]:
# Reference clip: cleaned exactly like the API does (Step 4a clean_clip), then F5's own trimming
# (<=12 s at a pause) + Whisper transcript of the trimmed audio, as run_f5 does.
import base64, numpy as np, soundfile as sf, librosa, torch
from types import SimpleNamespace
from IPython.display import Audio, display
from importlib.resources import files as pkg_files

USE_MY_CLIP = True
TEST_SENTENCE = "Hello, this is a quick test of my cloned voice. Does it sound like me?"   # = Step 4b

if USE_MY_CLIP:
    from google.colab import files as colab_files
    uploaded = colab_files.upload()
    REF_NAME, ref_bytes = next(iter(uploaded.items()))
else:
    REF_NAME = "basic_ref_en.wav (F5 example)"
    ref_bytes = open(str(pkg_files("f5_tts").joinpath("infer/examples/basic/basic_ref_en.wav")), "rb").read()
REF_B64 = base64.b64encode(ref_bytes).decode()

ref_clean = clean_clip(REF_B64, max_seconds=F5_MAX_REF_SECONDS)          # mono 24 kHz, as the API
trimmed, REF_TEXT = f5_utils.preprocess_ref_audio_text(ref_clean, "", show_info=lambda *a, **k: None)
REF_TEXT = REF_TEXT.strip()
# own copy: Step T5's forget_all_voices() deletes F5's cached trimmed clips
import shutil
REF_PATH = "/content/indicf5_reference.wav"
shutil.copy(trimmed, REF_PATH)
print(f"Reference: {REF_NAME}, trimmed to {sf.info(REF_PATH).duration:.1f}s at {sf.info(REF_PATH).samplerate} Hz")
print(f"Whisper transcript: {REF_TEXT!r}")
RESULTS["reference"] = {"name": REF_NAME, "seconds": round(sf.info(REF_PATH).duration, 1), "transcript": REF_TEXT}

def engine_check(engine):
    """One sentence through the exact API code (synthesize_core); returns a short result dict."""
    req = SimpleNamespace(text=TEST_SENTENCE, language="en", emotion="neutral", engine=engine,
                          stock_speaker=None, speaker_wav_base64=None, speaker_wav_base64_list=[REF_B64])
    try:
        wav, sr, extra = synthesize_core(req)
        display(Audio(wav, rate=sr))
        m = extra.get("metrics") or {}
        return {"ok": True, "synth_s": extra["synth_seconds"], "similarity": m.get("similarity")}
    except Exception as e:
        traceback.print_exc()
        return {"ok": False, "error": f"{type(e).__name__}: {e}"}

RESULTS["before"] = {}
for engine in ["xtts", "f5"]:
    print(f"\n== {engine} (before IndicF5) ==")
    RESULTS["before"][engine] = engine_check(engine)
    print(RESULTS["before"][engine])

## Step T3 — IndicF5, method A: `AutoModel.from_pretrained` (the official way) + compatibility fix
IndicF5's loader was written for its own copy of F5 and calls `load_model(...)` **without** a
checkpoint path (it loads the weights itself afterwards). Our f5-tts 1.1.22 requires that
argument, so for the duration of loading we wrap `load_model` to build the empty model only.
IndicF5 is an F5 **v0 ("Base")** model, so we also pass `text_mask_padding=False, pe_attn_head=1`,
exactly what f5-tts 1.1.22's own `F5TTS_Base.yaml` uses for v0 checkpoints.
The wrapper is removed again right after loading, so F5-TTS itself is never affected.

**Result (Python 3.13, torch 2.11): method A fails with a meta-tensor error. Use method B.**
Still run this cell: it defines helpers (`check_hindi_output`, `to_float`) that Steps T4 and T7 use.

In [ ]:
from contextlib import contextmanager
from transformers import AutoModel

HINDI_TEXT = "नमस्ते, मुझे थोड़ा पानी चाहिए। क्या आप मेरी मदद कर सकते हैं?"   # Hello, I need some water. Can you help me?
ORIGINAL_LOAD_MODEL = f5_utils.load_model
V0_ARCH = {"text_mask_padding": False, "pe_attn_head": 1}   # F5 v0 settings (F5TTS_Base.yaml)

def build_empty_cfm(model_cls, model_cfg, ckpt_path=None, **kwargs):
    """f5-tts 1.1.22's load_model, but with no checkpoint: IndicF5 loads its own weights."""
    if ckpt_path:                                  # normal F5 call: untouched
        return ORIGINAL_LOAD_MODEL(model_cls, model_cfg, ckpt_path, **kwargs)
    real_load_checkpoint = f5_utils.load_checkpoint
    f5_utils.load_checkpoint = lambda model, path, device, dtype=None, use_ema=True: model.to(device)
    try:
        return ORIGINAL_LOAD_MODEL(model_cls, {**V0_ARCH, **model_cfg}, "", **kwargs)
    finally:
        f5_utils.load_checkpoint = real_load_checkpoint

@contextmanager
def indicf5_compat():
    f5_utils.load_model = build_empty_cfm
    try:
        yield
    finally:
        f5_utils.load_model = ORIGINAL_LOAD_MODEL

def audio_report(wav, sr):
    """Length, loudness and peak: catches silent or pure-noise output."""
    wav = np.asarray(wav, dtype=np.float32).squeeze()
    rms = float(np.sqrt(np.mean(wav ** 2)) + 1e-9)
    rep = {"seconds": round(len(wav) / sr, 2), "sr": sr, "rms_dbfs": round(20 * np.log10(rms), 1),
           "peak": round(float(np.max(np.abs(wav))), 3)}
    rep["looks_ok"] = 0.8 < rep["seconds"] < 30 and -45 < rep["rms_dbfs"] < -6
    return rep

def check_hindi_output(label, wav, sr):
    """Plays the result, measures it, and lets Whisper write down what it hears (in Hindi)."""
    display(Audio(wav, rate=sr))
    path = f"/content/indicf5_{label}.wav"
    sf.write(path, wav, sr)
    rep = audio_report(wav, sr)
    try:
        rep["whisper_heard"] = f5_utils.transcribe(path, language="hi")
    except Exception as e:
        rep["whisper_heard"] = f"(failed: {e})"
    if spk_model is not None:   # cross-lingual similarity: English reference vs Hindi output
        ref16 = librosa.load(REF_PATH, sr=16000)[0]
        out16 = librosa.resample(np.asarray(wav, dtype=np.float32), orig_sr=sr, target_sr=16000)
        rep["similarity_to_ref"] = round(float(torch.nn.functional.cosine_similarity(
            speaker_embedding(ref16), speaker_embedding(out16), dim=0)), 3)
    print(rep)
    return rep

def to_float(audio):
    audio = np.asarray(audio)
    if audio.dtype == np.int16:
        audio = audio.astype(np.float32) / 32768.0
    return audio.astype(np.float32).squeeze()

indicf5_A = None
try:
    t0 = time.time()
    with indicf5_compat():
        indicf5_A = AutoModel.from_pretrained(INDICF5_REPO, trust_remote_code=True)
    assert f5_utils.load_model is ORIGINAL_LOAD_MODEL
    load_s = round(time.time() - t0, 1)
    print(f"Method A loaded in {load_s}s")
    t0 = time.time()
    wav_A = to_float(indicf5_A(HINDI_TEXT, ref_audio_path=REF_PATH, ref_text=REF_TEXT))
    gen_s = round(time.time() - t0, 1)
    print(f"Generated in {gen_s}s")
    RESULTS["method_A"] = {"ok": True, "load_s": load_s, "synth_s": gen_s, **check_hindi_output("A", wav_A, 24000)}
except Exception as e:
    traceback.print_exc()
    RESULTS["method_A"] = {"ok": False, "error": f"{type(e).__name__}: {str(e)[:400]}"}
    print("\nMethod A failed (that's OK for this test: run Step T4).")
print("GPU memory in use: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

## Step T4 — IndicF5, method B: load the weights directly (no remote code)
Same weights, but built with f5-tts 1.1.22's own functions, without running IndicF5's
`model.py`. Not affected by how `transformers` handles custom models, so it's our fallback
if method A fails or sounds wrong. Method A is unloaded first to free GPU memory.

In [ ]:
from collections import Counter
from safetensors.torch import load_file
from f5_tts.model import DiT

indicf5_A = None; torch.cuda.empty_cache()

def strip_prefix(state, prefix):
    """Keys starting with `prefix`, without it (and without torch.compile's '_orig_mod.')."""
    out = {}
    for k, v in state.items():
        k = k.replace("_orig_mod.", "")
        if k.startswith(prefix):
            out[k[len(prefix):]] = v
    return out

indicf5_B = None
try:
    t0 = time.time()
    weights_file = "model.safetensors" if "model.safetensors" in INDICF5_FILES else \
        next(f for f in INDICF5_FILES if f.endswith(".safetensors"))
    vocab_file = next(f for f in INDICF5_FILES if f.endswith("vocab.txt"))
    state = load_file(hf_hub_download(INDICF5_REPO, weights_file), device="cpu")
    vocab_path = hf_hub_download(INDICF5_REPO, vocab_file)
    print(f"{weights_file}: {len(state)} tensors, top-level names:",
          dict(Counter(k.replace('_orig_mod.', '').split('.')[0] for k in state)))

    # the TTS model (v0 architecture, same numbers IndicF5's model.py uses)
    cfm = build_empty_cfm(DiT, dict(dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4),
                          mel_spec_type="vocos", vocab_file=vocab_path, device="cpu")
    known_buffers = {"mel_spec.mel_stft.mel_scale.fb", "mel_spec.mel_stft.spectrogram.window"}
    cfm_state = {k: v for k, v in strip_prefix(state, "ema_model.").items() if k not in known_buffers}
    r = cfm.load_state_dict(cfm_state, strict=False)
    missing = [k for k in r.missing_keys if k not in known_buffers]
    print(f"TTS model: {len(cfm_state)} tensors loaded, missing {len(missing)}, unexpected {len(r.unexpected_keys)}",
          missing[:5], r.unexpected_keys[:5])

    # the vocoder (mel -> waveform); IndicF5 ships its own copy of the Vocos weights
    vocoder = f5_utils.load_vocoder("vocos", is_local=False, device="cpu")
    voc_state = strip_prefix(state, "vocoder.")
    if voc_state:
        rv = vocoder.load_state_dict(voc_state, strict=False)
        print(f"Vocoder: {len(voc_state)} tensors, missing {len(rv.missing_keys)}, unexpected {len(rv.unexpected_keys)}")
    else:
        print("Vocoder: no weights in the file, using standard Vocos 24 kHz")
    cfm = cfm.to(device).eval(); vocoder = vocoder.to(device).eval()
    indicf5_B = (cfm, vocoder)
    load_s = round(time.time() - t0, 1)
    del state

    t0 = time.time()
    wav_B, sr_B, _ = f5_utils.infer_process(REF_PATH, REF_TEXT, HINDI_TEXT, cfm, vocoder,
                                            mel_spec_type="vocos", speed=1.0, device=device,
                                            show_info=lambda *a, **k: None)
    gen_s = round(time.time() - t0, 1)
    print(f"Loaded in {load_s}s, generated in {gen_s}s")
    RESULTS["method_B"] = {"ok": True, "missing_keys": len(missing), "unexpected_keys": len(r.unexpected_keys),
                           "load_s": load_s, "synth_s": gen_s, **check_hindi_output("B", to_float(wav_B), sr_B)}
except Exception as e:
    traceback.print_exc()
    RESULTS["method_B"] = {"ok": False, "error": f"{type(e).__name__}: {str(e)[:400]}"}
print("GPU memory in use: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

## Step T5 — Did anything break? (XTTS, F5, Whisper, SpeechBrain, versions)
Clears the voice caches, then repeats Step T2's test and checks every other part.
F5's similarity naturally varies by about ±0.03 between runs.

In [ ]:
import subprocess
forget_all_voices()
RESULTS["after"] = {}
for engine in ["xtts", "f5"]:
    print(f"\n== {engine} (after IndicF5) ==")
    RESULTS["after"][engine] = engine_check(engine)
    print(RESULTS["after"][engine])

def check(name, fn):
    try:
        RESULTS["after"][name] = {"ok": True, "result": fn()}
    except Exception as e:
        traceback.print_exc()
        RESULTS["after"][name] = {"ok": False, "error": f"{type(e).__name__}: {e}"}
    print(name, "->", RESULTS["after"][name])

example = str(pkg_files("f5_tts").joinpath("infer/examples/basic/basic_ref_en.wav"))
check("xtts_stock", lambda: round(len(tts.tts(text="Stock voice check.", speaker="Ana Florence", language="en")) / SAMPLE_RATE, 2))
check("whisper", lambda: f5.transcribe(example))
check("speechbrain_similarity", lambda: round(float(torch.nn.functional.cosine_similarity(
    speaker_embedding(librosa.load(example, sr=16000)[0]),
    speaker_embedding(librosa.load(REF_PATH, sr=16000)[0]), dim=0)), 3))
check("speechbrain_accent", lambda: accent_scores(librosa.load(example, sr=16000)[0]))
check("load_model_restored", lambda: f5_utils.load_model is ORIGINAL_LOAD_MODEL)

after = versions()
RESULTS["versions_changed"] = {k: [VERSIONS_BEFORE[k], after[k]] for k in WATCH if after[k] != VERSIONS_BEFORE[k]}
print("\nVersions changed:", RESULTS["versions_changed"] or "none")
pc = subprocess.run("pip check 2>&1 | grep -iE 'transformers|coqui|tts|f5|librosa|numba|torch|cached|speechbrain|hyperpyyaml' "
                    "| grep -viE '^(diffusers|gradio)'", shell=True, capture_output=True, text=True).stdout.strip()
RESULTS["pip_check"] = pc or "no conflicts in the packages ReVoice uses"
print("pip check:", RESULTS["pip_check"])

## Step T6 — Copy this output back to Claude
Also listen to the two Hindi results (Steps T3 and T4) and say how they sound:
clear Hindi? your voice? any mumbling?

In [ ]:
def verdict():
    ok_before = all(RESULTS.get("before", {}).get(e, {}).get("ok") for e in ["xtts", "f5"])
    ok_after = all(v.get("ok") for v in RESULTS.get("after", {}).values())
    a = RESULTS.get("method_A", {}); b = RESULTS.get("method_B", {})
    return {"existing_engines_ok_before": ok_before,
            "existing_engines_ok_after": ok_after,
            "load_model_restored": RESULTS.get("after", {}).get("load_model_restored", {}).get("result"),
            "no_versions_changed": not RESULTS.get("versions_changed"),
            "indicf5_method_A": a.get("ok") and a.get("looks_ok"),
            "indicf5_method_B": b.get("ok") and b.get("looks_ok")}
RESULTS["verdict"] = verdict()
print("=" * 30, "PASTE FROM HERE", "=" * 30)
print(json.dumps(RESULTS, indent=1, ensure_ascii=False, default=str))
print("=" * 30, "PASTE TO HERE", "=" * 32)

## Step T7 — Remove the garbled start of the Hindi sentence (method B, ~5 minutes)
**Cause found in f5-tts 1.1.22's code** (`utils_infer.infer_batch_process`): with no duration given,
F5 makes the new speech `reference seconds × (bytes of Hindi text ÷ bytes of reference transcript)` long.
It counts **bytes**, not letters: a Devanagari letter is 3 bytes in UTF-8, an English letter 1 byte. So
with an English transcript the Hindi part gets about **3× too much time**, and the model fills the extra
time with mumbling before the real sentence. T7 also prints how much of the English transcript IndicF5's
vocabulary even knows (unknown characters are read as a blank).

Each setting runs with 2 random seeds (F5 starts from random noise), at nfe_step 32 like Step T4:
0. **baseline** = Step T4 again
1. **explicit duration**: Hindi syllables ÷ 5 per second + short pauses at punctuation; F5 gets the
   total (reference + Hindi) instead of its own guess
2. **shorter reference**: cut at a pause between 6 and 8 s, Whisper re-transcribes what's left
3. **Devanagari transcript** of the English reference (by sound, not translated): Gemini if a
   `GEMINI_API_KEY` secret exists (optional: key icon → same key as `backend/.env`), else rough spelling rules
4. **combination** of whatever beat the baseline (and all three together, to compare)

Every output is **peak-normalised to -1 dBFS** (no clipping). Finally the best setting runs again at
**nfe_step 64** (quality vs time). For each run: length, Whisper's Hindi transcript, similarity to your
reference, `junk_before` (letters heard before the sentence starts) and `cer` (wrong letters, 0 = perfect).

Needs Steps T3 and T4 from this session. Listen to the players, then copy the printed block back to Claude.

In [ ]:
# Step T7: finds a setting without the garbled start. Uses method B (Step T4) only.
import re, difflib, unicodedata, requests

if globals().get("indicf5_B") is None:
    raise RuntimeError("IndicF5 method B isn't loaded. Run Step T4 first (after Steps 2b-4a, T1, T2).")
cfm, vocoder = indicf5_B

NFE_STEPS = 32                  # T4 used f5-tts's default (32); the last test also tries 64
SEEDS = [0, 1]                  # F5 starts from random noise: 2 seeds per setting, so one lucky run doesn't decide
PEAK_DBFS = -1.0                # every output is peak-normalised to -1 dBFS (removes clipping)
HINDI_SYLLABLES_PER_SEC = 5.0   # normal Hindi speaking rate
COMMA_PAUSE_S = 0.25            # pause after , ; :
SENTENCE_PAUSE_S = 0.4          # pause after । . ? ! inside the text (not at the very end)
SHORT_REF = (6.0, 8.0)          # option 2: cut the reference at a pause between 6 and 8 seconds
HOP, SR = f5_utils.hop_length, f5_utils.target_sample_rate   # 256 samples at 24 kHz = one F5 frame (10.7 ms)

# ---------- 1. explicit duration from the Hindi text ----------
CONSONANT = re.compile("[क-हक़-य़]")     # क ... ह, plus क़ ... य़
VOWEL_LETTER = re.compile("[ऄ-औॠॡ]")   # अ ... औ (a vowel written on its own)
VOWEL_SIGN = re.compile("[ा-ौॢॣ]")     # ा ि ी ु ू े ै ो ौ ...
VIRAMA, NUKTA = "्", "़"                          # ् joins two consonants; ़ is a dot (ड़)

def hindi_syllables(text):
    """Counts syllables: each vowel letter, and each consonant that carries a vowel.
    A consonant with ् (virama) joins the next one (no syllable of its own), and a bare
    consonant at the END of a word doesn't count either, because Hindi drops that final
    'a' (आप = 'aap', 1 syllable; मदद = 'madad', 2)."""
    count = 0
    for word in re.findall("[ऀ-ॿ]+", text):
        letters = word.replace(NUKTA, "")
        for i, ch in enumerate(letters):
            nxt = letters[i + 1] if i + 1 < len(letters) else ""
            if VOWEL_LETTER.match(ch):
                count += 1
            elif CONSONANT.match(ch):
                if nxt == VIRAMA:
                    continue                    # part of a cluster (स्त in नमस्ते)
                if nxt == "" and i > 0:
                    continue                    # bare last consonant: the final 'a' is dropped
                count += 1                      # carries a vowel (written, or the inherent 'a')
    return count

def estimate_hindi_seconds(text):
    """Syllables at a normal speaking rate, plus short pauses at punctuation."""
    body = text.strip().rstrip("।.?!,;: ")      # no pause after the last word
    pauses = sum(COMMA_PAUSE_S for c in body if c in ",;:") + sum(SENTENCE_PAUSE_S for c in body if c in "।.?!")
    return round(hindi_syllables(text) / HINDI_SYLLABLES_PER_SEC + pauses, 2)

def text_batches(ref_path, ref_text, gen_text):
    """How infer_process splits the text (same formula as f5-tts 1.1.22)."""
    ref_s = sf.info(ref_path).duration
    return f5_utils.chunk_text(gen_text, max_chars=int(len(ref_text.encode("utf-8")) / ref_s * (22 - ref_s)))

def default_gen_seconds(ref_path, ref_text, gen_text):
    """F5's own guess when no duration is given (infer_batch_process): reference length x
    (gen_text BYTES / ref_text BYTES). Devanagari letters are 3 bytes in UTF-8, English ones 1,
    so with an English ref_text the Hindi part gets about 3x too much time."""
    if len(ref_text[-1].encode("utf-8")) == 1:
        ref_text += " "                          # F5 adds this space itself
    ref_frames = int(sf.info(ref_path).duration * SR) // HOP
    frames = sum(int(ref_frames / len(ref_text.encode("utf-8")) * len(b.encode("utf-8")))
                 for b in text_batches(ref_path, ref_text, gen_text))
    return round(frames * HOP / SR, 2)

# ---------- 2. shorter reference ----------
def shorten_reference(path, lo, hi):
    """Cuts the reference in the middle of a pause between lo and hi seconds (nearest the
    middle of that range), then Whisper writes down exactly what is left, so the transcript
    matches the audio."""
    wav, sr = sf.read(path, dtype="float32")
    total = len(wav) / sr
    if total <= hi:
        return path, f"already {total:.1f}s: not cut"
    speech = librosa.effects.split(wav, top_db=35, frame_length=1024, hop_length=256)
    pauses = [(e1 + s2) / 2 / sr for (_, e1), (s2, _) in zip(speech[:-1], speech[1:]) if (s2 - e1) / sr >= 0.1]
    inside = [p for p in pauses if lo <= p <= hi]
    if inside:
        cut, how = min(inside, key=lambda p: abs(p - (lo + hi) / 2)), "at a pause"
    else:
        cut, how = hi, "no pause in range: cut mid-speech"
    out = "/content/indicf5_reference_short.wav"
    sf.write(out, wav[:int(cut * sr)], sr)
    return out, f"cut at {cut:.1f}s of {total:.1f}s ({how})"

# ---------- 3. reference transcript in Devanagari ----------
ROUGH_VOWELS = {    # English spelling -> (vowel at the start of a word, vowel sign after a consonant)
    "ee": ("ई", "ी"), "ea": ("ई", "ी"), "oo": ("ऊ", "ू"), "ai": ("ए", "े"), "ay": ("ए", "े"),
    "oa": ("ओ", "ो"), "ou": ("औ", "ौ"), "ow": ("औ", "ौ"), "oi": ("ऑय", "ॉय"), "oy": ("ऑय", "ॉय"),
    "a": ("अ", ""), "e": ("ए", "े"), "i": ("इ", "ि"), "o": ("ओ", "ो"), "u": ("उ", "ु")}
ROUGH_CONSONANTS = {
    "tch": "च", "sh": "श", "ch": "च", "th": "थ", "ph": "फ", "ck": "क", "qu": "क्व", "x": "क्स",
    "b": "ब", "c": "क", "d": "ड", "f": "फ", "g": "ग", "h": "ह", "j": "ज", "k": "क", "l": "ल", "m": "म",
    "n": "न", "p": "प", "q": "क", "r": "र", "s": "स", "t": "ट", "v": "व", "w": "व", "y": "य", "z": "ज़"}

def rough_word(w):
    """One English word written in Devanagari by simple spelling rules (only approximate:
    English spelling isn't phonetic)."""
    w = re.sub(r"c(?=[eiy])", "s", w)                        # soft c: "voice" -> "voise"
    w = re.sub(r"([b-df-hj-np-tv-z])\1", r"\1", w)            # double letters: "hello" -> "helo"
    if len(w) > 3 and w.endswith("e") and w[-2] not in "aeiou":
        w = w[:-1]                                           # silent final e: "like" -> "lik"
    out, after_consonant, i = "", False, 0
    while i < len(w):
        for n in (3, 2, 1):
            piece = w[i:i + n]
            if len(piece) == n and (piece in ROUGH_VOWELS or piece in ROUGH_CONSONANTS):
                break
        else:
            i += 1; continue                                 # a letter we have no rule for
        if piece == "y" and i > 0 and w[i + 1:i + 2] not in tuple("aeiou"):
            piece_vowel = ("ई", "ी")                          # "y" as a vowel: "my", "happy"
        else:
            piece_vowel = ROUGH_VOWELS.get(piece)
        if piece_vowel:
            out += piece_vowel[1] if after_consonant else piece_vowel[0]
            after_consonant = False
        else:
            out += (VIRAMA if after_consonant else "") + ROUGH_CONSONANTS[piece]   # ् joins two consonants
            after_consonant = True
        i += n
    return out

def rough_devanagari(text):
    parts = re.findall(r"[A-Za-z']+|[^A-Za-z']+", text)
    return "".join(rough_word(p.lower().replace("'", "")) if p[0].isalpha() or p[0] == "'"
                   else p.replace(".", "।") for p in parts)

def gemini_devanagari(text):
    """Asks Gemini to write the English sentence in Devanagari letters as it SOUNDS
    (not a translation). Returns None if there is no GEMINI_API_KEY secret or it fails."""
    try:
        key = userdata.get("GEMINI_API_KEY")
        prompt = ("Transliterate this English sentence into Devanagari script exactly as it SOUNDS "
                  "when spoken. Do NOT translate it. Keep every word in order and keep the punctuation. "
                  "Reply with the Devanagari text only.\n\n" + text)
        r = requests.post("https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent",
                          json={"contents": [{"parts": [{"text": prompt}]}], "generationConfig": {"temperature": 0}},
                          headers={"x-goog-api-key": key}, timeout=60)
        r.raise_for_status()
        out = r.json()["candidates"][0]["content"]["parts"][0]["text"].strip()
    except Exception as e:
        print(f"   Gemini not used ({type(e).__name__}: {str(e)[:120]})")
        return None
    letters = [c for c in out if c.isalpha() or unicodedata.category(c).startswith("M")]
    if not letters or sum("ऀ" <= c <= "ॿ" for c in letters) / len(letters) < 0.9:
        print(f"   Gemini reply isn't Devanagari, not used: {out[:80]!r}")
        return None
    return out

devanagari_cache = {}
def to_devanagari(english):
    """Gemini if a GEMINI_API_KEY secret exists, otherwise the rough spelling rules."""
    if english not in devanagari_cache:
        dev = gemini_devanagari(english)
        devanagari_cache[english] = (dev, "gemini") if dev else (rough_devanagari(english), "rough rules")
    return devanagari_cache[english]

def finish_ref_text(text):
    """Ends the transcript with a full stop (। for Devanagari) and a space, so it doesn't run
    into the Hindi text. F5 adds the space itself after . ? ! but not after ।"""
    text = text.strip()
    if not text.endswith((".", "?", "!", "।")):
        text += "।" if re.search("[ऀ-ॿ]", text) else "."
    return text + " " if text.endswith("।") else text

def vocab_known(text):
    """Share of the transcript's characters that are in IndicF5's vocabulary. Unknown ones are
    read as token 0 (a blank), so an unknown transcript tells the model almost nothing."""
    chars = [c for c in f5_utils.convert_char_to_pinyin([text])[0] if c != " "]
    return round(sum(c in cfm.vocab_char_map for c in chars) / max(len(chars), 1), 2)

# ---------- checking each result ----------
def plain_letters(text):
    """Letters only (no spaces or punctuation like , । ?), so we compare words, not punctuation."""
    return "".join(c for c in unicodedata.normalize("NFC", text)
                   if not c.isspace() and not unicodedata.category(c).startswith(("P", "Z", "S")))

def edit_distance(a, b):
    """Fewest letter changes (insert, delete, replace) to turn a into b."""
    row = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, row[0] = row[0], i
        for j, cb in enumerate(b, 1):
            prev, row[j] = row[j], min(row[j] + 1, row[j - 1] + 1, prev + (ca != cb))
    return row[-1]

def garble_check(heard, expected):
    """junk_before = letters Whisper heard BEFORE the real sentence starts (the garbled start);
    cer = character error rate against the sentence we asked for (0 = perfect)."""
    h, e = plain_letters(heard), plain_letters(expected)
    blocks = [b for b in difflib.SequenceMatcher(None, h, e, autojunk=False).get_matching_blocks() if b.size >= 3]
    junk = max(0, blocks[0].a - blocks[0].b) if blocks else len(h)   # minus letters of the sentence Whisper missed
    return {"junk_before": junk, "prefix_gone": junk <= 2, "cer": round(edit_distance(h, e) / len(e), 2)}

def peak_normalise(wav, dbfs=PEAK_DBFS):
    """Scales the audio so its loudest sample is at -1 dBFS (0.891): no clipping when saved."""
    peak = float(np.max(np.abs(wav)))
    return (wav * (10 ** (dbfs / 20) / peak) if peak > 0 else wav), peak

def synth(ref_path, ref_text, total_s, nfe, seed):
    torch.manual_seed(seed)                     # same random start for every setting = fair comparison
    torch.cuda.synchronize(); t0 = time.time()
    wav, sr, _ = f5_utils.infer_process(ref_path, ref_text, HINDI_TEXT, cfm, vocoder, mel_spec_type="vocos",
                                        speed=1.0, nfe_step=nfe, fix_duration=total_s, device=device,
                                        show_info=lambda *a, **k: None, progress=None)
    torch.cuda.synchronize()
    return to_float(wav), sr, round(time.time() - t0, 2)

def mean(values):
    values = [v for v in values if v is not None]
    return round(sum(values) / len(values), 3) if values else None

T7 = {"hindi_text": HINDI_TEXT, "target_hindi_s": estimate_hindi_seconds(HINDI_TEXT),
      "hindi_syllables": hindi_syllables(HINDI_TEXT), "settings": {}}

def run_setting(name, ref_path, ref_text, explicit, nfe=NFE_STEPS):
    """Generates the Hindi sentence once per seed and measures every result."""
    ref_text = finish_ref_text(ref_text)
    ref_s = sf.info(ref_path).duration
    total_s = round(ref_s + T7["target_hindi_s"], 2) if explicit else None     # F5 wants ref + new speech
    runs = []
    for seed in SEEDS:
        print(f"\n===== {name} | seed {seed} =====")
        wav, sr, synth_s = synth(ref_path, ref_text, total_s, nfe, seed)
        wav, raw_peak = peak_normalise(wav)
        rep = check_hindi_output(f"T7_{name}_seed{seed}", wav, sr)      # plays it, Whisper, similarity
        runs.append({"seed": seed, "seconds": rep["seconds"], "synth_s": synth_s,
                     "synth_per_audio_s": round(synth_s / rep["seconds"], 2), "raw_peak": round(raw_peak, 3),
                     "similarity": rep.get("similarity_to_ref"), "heard": rep["whisper_heard"],
                     **garble_check(rep["whisper_heard"], HINDI_TEXT)})
    T7["settings"][name] = {
        "ref_file": ref_path, "ref_s": round(ref_s, 1), "ref_text": ref_text.strip(), "ref_vocab_known": vocab_known(ref_text),
        "explicit_duration": explicit, "duration": f"explicit: {total_s}s total ({T7['target_hindi_s']}s Hindi)" if explicit
                    else f"F5 default: ~{default_gen_seconds(ref_path, ref_text, HINDI_TEXT)}s Hindi",
        "text_batches": len(text_batches(ref_path, ref_text, HINDI_TEXT)), "nfe_step": nfe,
        "prefix_gone_all_seeds": all(r["prefix_gone"] for r in runs),
        "mean_cer": mean(r["cer"] for r in runs), "mean_similarity": mean(r["similarity"] for r in runs),
        "mean_seconds": mean(r["seconds"] for r in runs), "mean_synth_s": mean(r["synth_s"] for r in runs),
        "runs": runs}
    s = T7["settings"][name]
    print(f"\n>>> {name}: garbled start gone in all seeds = {s['prefix_gone_all_seeds']}, CER {s['mean_cer']}, "
          f"similarity {s['mean_similarity']}, {s['mean_seconds']}s audio in {s['mean_synth_s']}s")
    return s

# the reference variants
short_path, short_how = shorten_reference(REF_PATH, *SHORT_REF)
short_text = REF_TEXT if short_path == REF_PATH else f5_utils.transcribe(short_path)
dev_full, dev_source = to_devanagari(REF_TEXT)
T7["short_reference"] = {"how": short_how, "transcript": short_text}
T7["devanagari"] = {"source": dev_source, "text": dev_full}
print(f"Hindi target: {T7['hindi_syllables']} syllables -> {T7['target_hindi_s']}s")
print(f"Short reference: {short_how}; transcript {short_text!r}")
print(f"Devanagari reference text ({dev_source}): {dev_full!r}")
print(f"IndicF5 knows {vocab_known(REF_TEXT):.0%} of the English transcript's characters, "
      f"{vocab_known(dev_full):.0%} of the Devanagari one's")

base = run_setting("0_baseline_T4", REF_PATH, REF_TEXT, explicit=False)
opts = {"explicit_duration": run_setting("1_explicit_duration", REF_PATH, REF_TEXT, explicit=True),
        "short_reference": run_setting("2_short_reference", short_path, short_text, explicit=False),
        "devanagari_text": run_setting("3_devanagari_ref_text", REF_PATH, dev_full, explicit=False)}

def helped(s):
    """Better than the baseline: the garbled start is gone, or clearly fewer wrong letters."""
    return (s["prefix_gone_all_seeds"] and not base["prefix_gone_all_seeds"]) or s["mean_cer"] < base["mean_cer"] - 0.05

def combination(parts):
    """Runs the chosen options together; returns the setting's name."""
    path, text = (short_path, short_text) if "short_reference" in parts else (REF_PATH, REF_TEXT)
    if "devanagari_text" in parts:
        text = to_devanagari(text)[0]
    name = "4_" + "+".join(parts)
    run_setting(name, path, text, explicit="explicit_duration" in parts)
    return name

best_parts = [k for k, s in opts.items() if helped(s)] or list(opts)   # nothing helped alone: try all three
T7["combination_of"] = best_parts
combos = [combination(best_parts)]
if len(best_parts) < 3:
    combos.append(combination(list(opts)))                             # and all three, for comparison

def rank(name):
    s = T7["settings"][name]
    return (not s["prefix_gone_all_seeds"], s["mean_cer"], -(s["mean_similarity"] or 0))
best = min(T7["settings"], key=rank)

# speed vs quality: the best setting again with 64 steps (the ReVoice server's F5 uses 64)
best_s = T7["settings"][best]
run_setting(best + "_nfe64", best_s["ref_file"], best_s["ref_text"], explicit=best_s["explicit_duration"], nfe=64)
T7["best"] = best
T7["nfe_32_vs_64"] = {n: {k: T7["settings"][n][k] for k in ("mean_cer", "mean_similarity", "mean_seconds", "mean_synth_s")}
                      for n in (best, best + "_nfe64")}
RESULTS["T7"] = T7

print("\n" + "=" * 30, "PASTE FROM HERE", "=" * 30)
print(json.dumps(T7, indent=1, ensure_ascii=False, default=str))
print("=" * 30, "PASTE TO HERE", "=" * 32)
print(f"\nBest setting by the numbers: {best}. Listen to it and to {best}_nfe64 above before deciding.")

## If something fails
- **Step T1 "Access not granted"**: accept the terms on the IndicF5 page (logged in with the same
  account as the token), wait a minute, re-run Step T1.
- **Method A fails but B works**: fine; paste the output. ReVoice can use method B.
- **An error mentions `accelerate`**: run `!pip install -q --no-deps accelerate`, then re-run
  Step T3 (no restart needed). Tell Claude, so it goes into the documented install lines.
- **Out of GPU memory**: Runtime → Restart session, run Steps 2b → 4a, T1, T2, T3 (its helpers are
  needed; method A failing is expected), then T4 and T5 (or T7).
- Never `pip install git+https://github.com/ai4bharat/IndicF5` here: it replaces f5-tts (see top).
- **Step T7 says "Gemini not used"**: fine, it falls back to rough spelling rules. For a better
  Devanagari transcript, add a `GEMINI_API_KEY` secret (same key as `backend/.env`, Notebook access on)
  and re-run T7.